# Parent Dataset Loading

## Imports

In [18]:
import psycopg2
from tabulate import tabulate
#import a folder in the parent directory
import sys
sys.path.append('../')
import dbtools.dbtools as qrs
import dbtools.load as load
import pandas as pd
from pathlib import Path

## Connection

In [19]:
try:
    # Connect to the PostgreSQL database
    conn = qrs.connect()
    print("Connected to the database")

except (Exception, psycopg2.DatabaseError) as error:
    print(error)

Connected to the database


## Datasettype

In [20]:
datasettpypes_table = qrs.get_data('datasettypes')
datasettpypes_table['description_datasettype'].tolist()

['2024 Dataset Type. Material mask computed using otsu.Onlypores computed using otsu. Patching without padding, cropping the images to a size divisible by the patch size.Patches with less than 80% of the material are discarded.Rows are in this format: - Last column is the areafrac: To compute this value, the  3D patch is projected in Z to 2D and the pore are is divided by the material area- Second last column is the volfrac: The pore volume is divided by the total volume of the patchIn both division a epislon of 1e-6 is added to avoid division by zero.',
 '2025 Dataset Type. Material mask computed using otsu.Onlypores computed using sauvola  radius 30 k=0.125 r=128. Onlypores computed only between the frontwall and backwall, both calculated using aligner.crop_walls(). Pore Filtering min size 8 and min pixels in each axes 2.Patching without padding, cropping the images to a size divisible by the patch size.Patches with less than 80% of the material are discarded.Rows are in this format:

In [21]:
datasettype = 6

## Children

Pick the file_path values of the datasets that the new parent dataset should contain. They must already exist in the `datasets` table.

In [22]:
# List existing datasets to pick children from
datasets_table = qrs.get_data('datasets')
datasets_table[['id_dataset', 'file_path_dataset']]

,id_dataset,file_path_dataset
0,921,\\192.168.10.106\imdea\DataDriven_UT_AlbertoVi...
1,922,\\192.168.10.106\imdea\DataDriven_UT_AlbertoVi...
2,923,\\192.168.10.106\imdea\DataDriven_UT_AlbertoVi...
3,924,\\192.168.10.106\imdea\DataDriven_UT_AlbertoVi...
4,925,\\192.168.10.106\imdea\DataDriven_UT_AlbertoVi...
...,...,...
762,1226,/home/alberto.vicente/phd/DataDriven_UT_Albert...
763,1227,/home/alberto.vicente/phd/DataDriven_UT_Albert...
764,1228,/home/alberto.vicente/phd/DataDriven_UT_Albert...
765,1229,/home/alberto.vicente/phd/DataDriven_UT_Albert...


In [23]:
child_dataset_paths = [str(Path(path)) for path in [
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_4/registration_48/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_5/registration_49/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_7/registration_50/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_8/registration_51/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_10/registration_52/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_11/registration_53/image_manifest.csv',
    '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/JI_12/registration_54/image_manifest.csv'
]]

## Parent dataset

In [24]:
parent_dataset_path = str(Path(
    r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/image_dataset_manifest.csv'
))

patch_size = '1'
registration_id = []
targets = ['Volfrac']
reconstruction_shape = (-1, -1)
description = 'Parent dataset grouping the datasets listed in child_dataset_paths.'

# rows can be computed from the parent file itself if it exists, or set
# manually to the total number of rows across the children it groups
rows = sum(len(pd.read_csv(path)) for path in child_dataset_paths)

## Load the parent dataset and link its children

In [25]:
parent_id = load.load_parent_dataset(
    conn=conn,
    datasettype_id=datasettype,
    file_path=parent_dataset_path,
    rows=rows,
    patch_size=patch_size,
    targets=targets,
    reconstruction_shape=reconstruction_shape,
    registration_ids=registration_id,
    child_dataset_paths=child_dataset_paths,
    description=description,
)

parent_id

Parent dataset from '/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/04_ML_data/Juan Ignacio/2026/5MHz/image_dataset_manifest.csv' loaded with ID: 1234


1234

## Make sure insert was correct

In [26]:
# Should list exactly the datasets in child_dataset_paths
qrs.get_dataset_children(parent_dataset_path)

,id_dataset,file_path_dataset,description_dataset,datasettype_id_dataset,rows_dataset,patch_size_dataset,reconstruction_shape_dataset,target_dataset,image_manifest_path_dataset,ut_image_path_dataset,...,depth_roi_end_exclusive_from_frontwall_dataset,depth_roi_end_inclusive_from_frontwall_dataset,depth_roi_start_z_dataset,depth_roi_end_exclusive_z_dataset,depth_roi_end_inclusive_z_dataset,material_frontwall_z_dataset,material_backwall_z_dataset,material_threshold_dataset,xct_resolution_dataset,ut_resolution_dataset
760,1224,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,9120 cardinal,no_ut_patches pixels,"(3, 80, 38) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,1225,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,9234 cardinal,no_ut_patches pixels,"(3, 81, 38) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
762,1226,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,10332 cardinal,no_ut_patches pixels,"(3, 82, 42) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
763,1227,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,10458 cardinal,no_ut_patches pixels,"(3, 83, 42) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
764,1228,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,9360 cardinal,no_ut_patches pixels,"(3, 80, 39) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
765,1229,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,9720 cardinal,no_ut_patches pixels,"(3, 81, 40) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
766,1230,/home/alberto.vicente/phd/DataDriven_UT_Albert...,"Image-shaped dataset without UT patches, creat...",7,9600 cardinal,no_ut_patches pixels,"(3, 80, 40) pixels tuple",areafrac nominal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
conn.close()